# Additional checks on existing outputs

Four checks that answer likely examiner questions, all computed from the results already in `dimsmauls/code4ml-eval-outputs` (no generation, no rating):

| Section | Check | Bab IV |
|---|---|---|
| A6 | Pass@1 and Pass@10 per DS-1000 library | 4.5 |
| A3 | How often code extraction used the first python block instead of the official DS-1000 post-processing | 4.3 |
| A2 | Failure taxonomy of vanilla 7B vs Qwen 14B on DS-1000 | 4.6 |
| A4 | Compute-matched baseline for the refinement loop: Pass@4 without the loop on the same 200 problems | 4.7 |

**Colab:** add a Hugging Face token with read access as the secret `HF_TOKEN`, then Run all. Locally an existing `hf auth login` is used.

In [ ]:
%pip install -q pandas huggingface_hub   # into the selected kernel (Colab or local)

In [ ]:
import os

try:  # Colab secret; elsewhere an existing `hf auth login` is used
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

import json
import re
from math import comb, sqrt

import pandas as pd
from huggingface_hub import hf_hub_download

pd.set_option("display.width", 200)

REPO = "dimsmauls/code4ml-eval-outputs"
SEEDS = (42, 43, 44)
RUNS = {"vanilla": ["vanilla"], "few-shot": ["few-shot"], "qwen-14b": ["qwen-14b"],
        "lora": [f"lora/seed-{s}" for s in SEEDS], "qlora": [f"qlora/seed-{s}" for s in SEEDS]}
ARM_ORDER = ["vanilla", "few-shot", "lora", "qlora", "qwen-14b"]
REFINEMENT_RUNS = {("execution", m): f"refinement-{m}/seed-42" for m in ("lora", "qlora")}
REFINEMENT_RUNS.update({("execution + SonarQube", m): f"refinement-sonar-{m}/seed-42" for m in ("lora", "qlora")})


def get(path):
    return hf_hub_download(REPO, path, repo_type="dataset")


summaries = {run: json.load(open(get(f"{run}/summary.json"))) for runs in RUNS.values() for run in runs}

## A6. Pass@1 and Pass@10 per library

`by_library` from each run's `summary.json` (unbiased estimator); LoRA and QLoRA = mean of the 3 seeds.

In [ ]:
libraries = sorted(summaries["vanilla"]["ds1000"]["by_library"])
rows = []
for lib in libraries:
    row = {"library": lib, "problems": summaries["vanilla"]["ds1000"]["by_library"][lib]["problems"]}
    for k in ("pass@1", "pass@10"):
        for arm in ARM_ORDER:
            values = [summaries[run]["ds1000"]["by_library"][lib][k] for run in RUNS[arm]]
            row[f"{k} {arm}"] = sum(values) / len(values)
    rows.append(row)
per_library = pd.DataFrame(rows).set_index("library")
per_library.round(3)

## A3. Code extraction path per run

`evalkit.ds1000_code` keeps the official DS-1000 post-processing when its result parses and otherwise post-processes the first python block. A sample used the python-block path when its stored `code` differs from the official post-processing of its `completion`.

In [ ]:
def official_postprocess(code):
    """Official DS-1000 post-processing (test_ds1000.py, commit b39aab7), same as evalkit.postprocess_ds1000."""
    code = code.split("</code>")[0]
    code = code.replace("```python", "")
    code = code.split("```")[0]
    code = code.split("\nEND SOLUTION")[0]
    return code.replace("<code>", "")


generations = {run: pd.read_json(get(f"{run}/ds1000_generations.jsonl"), lines=True)
               for runs in RUNS.values() for run in runs}
extraction = pd.DataFrame([
    {"run": run, "samples": len(g),
     "python_block_samples": int((g["code"] != g["completion"].map(official_postprocess)).sum())}
    for run, g in generations.items()]).set_index("run")
extraction["python_block_%"] = 100 * extraction["python_block_samples"] / extraction["samples"]
extraction.round(1)

## A2. Failure taxonomy: vanilla 7B vs Qwen 14B

Per DS-1000 sample: test outcome category, whether the answer re-creates sample input data (`df = pd.DataFrame(...)`, `np.array(...)`, `tf.constant(...)`, `torch.tensor(...)` on a common input name, which overwrites the test's own input), and whether the reply was cut at `max_new_tokens` (odd number of code fences). The re-creation pattern is a regex heuristic on common variable names, so it is a lower bound.

In [ ]:
REDEFINE = r"(df|data|a|x|X|A|B|s|series) *= *(pd\.DataFrame|pd\.Series|np\.array|tf\.constant|torch\.tensor)\("
SYNTAX = r"Syntax|indent|unterminated|invalid (syntax|decimal|character)|never closed"


def failure_type(result, passed):
    if passed:
        return "passed"
    if result == "timed out":
        return "timed out"
    if re.fullmatch(r"failed: *", result):
        return "wrong answer (assertion)"
    if re.search(SYNTAX, result):
        return "syntax error"
    if "is not defined" in result:
        return "NameError"
    return "other runtime error"


taxonomy = {}
for run in ("vanilla", "qwen-14b"):
    results = pd.read_json(get(f"{run}/ds1000_results.jsonl"), lines=True)
    df = generations[run].merge(results, on=["problem_id", "sample"])
    df["type"] = [failure_type(r, p) for r, p in zip(df["result"], df["passed"])]
    df["recreates_sample_data"] = df["code"].str.contains(REDEFINE, regex=True)
    df["truncated"] = df["completion"].str.count("```") % 2 == 1
    taxonomy[run] = df

display(pd.DataFrame({run: df["type"].value_counts() for run, df in taxonomy.items()}).fillna(0).astype(int))
display(pd.DataFrame({run: {"recreates sample data %": 100 * df["recreates_sample_data"].mean(),
                            "truncated %": 100 * df["truncated"].mean(),
                            "mean reply length (chars)": df["completion"].str.len().mean()}
                      for run, df in taxonomy.items()}).round(1))
display(pd.concat({run: df.groupby(["recreates_sample_data", "truncated"])["passed"].agg(n="size", pass_rate="mean")
                   for run, df in taxonomy.items()}).round(3))

## A4. Refinement loop vs Pass@4 without the loop

The loop gives each problem up to 4 greedy attempts (iteration 0-3). A compute-matched baseline is Pass@4 on the same 200 problems from the 10 sampled answers of the same adapter seed (unbiased estimator). Difference = solved by the loop (0/1) minus Pass@4, per problem; 95% CI by normal approximation.

In [ ]:
def pass_at_k(n, c, k):
    return 1.0 if n - c < k else 1 - comb(n - c, k) / comb(n, k)


rows = []
for (loop, method), path in REFINEMENT_RUNS.items():
    summary = json.load(open(get(f"{path}/summary.json")))
    loop_rows = pd.read_json(get(f"{path}/refinement_results.jsonl"), lines=True)
    solved = set(loop_rows.loc[loop_rows["passed"], "problem_id"])
    sampled = pd.read_json(get(f"{method}/seed-{summary['seed']}/ds1000_results.jsonl"), lines=True)
    counts = sampled.groupby("problem_id")["passed"].agg(["size", "sum"])
    diff = pd.Series([(pid in solved) - pass_at_k(int(counts.loc[pid, "size"]), int(counts.loc[pid, "sum"]), 4)
                      for pid in summary["problem_ids"]])
    se = diff.std(ddof=1) / sqrt(len(diff))
    rows.append({"loop": f"{method} | {loop}", "problems": len(diff),
                 "refinement (<= 4 greedy attempts)": len(solved) / len(diff),
                 "pass@4 without loop": (len(solved) / len(diff)) - diff.mean(),
                 "diff": diff.mean(), "ci95_low": diff.mean() - 1.96 * se, "ci95_high": diff.mean() + 1.96 * se})
pd.DataFrame(rows).set_index("loop").round(4)